# Primitive Rateless: an interactive soft-XOR laboratory

The default sparse polynomial is **$P(x)=x^5+x^2+1$**, with **$K=5$** and **$N=16$**. Polynomial exponents are the only code definition: change `P` below, and `K` follows automatically. We trust your choice of a primitive polynomial; no period enumeration or primality/primitivity check is performed.

The workflow has three independent stages:

1. Run the setup and editor cells, then edit the observation for as long as you like.
2. Run **Decode the current observation** to copy the current inputs and run the selected decoders head-to-head for $T$ iterations.
3. Run the plot cells: each cell has one chart with decoder and metric/bit selectors.

Install with `uv sync --group dev` from the repository root and use the project's Python kernel in VS Code or JupyterLab.

In [1]:
import sys
from pathlib import Path

notebooks_dir = (
    Path.cwd() if (Path.cwd() / "_shared").is_dir() else Path.cwd() / "notebooks"
)
if not (notebooks_dir / "_shared").is_dir():
    raise RuntimeError("Start the kernel in the repository root or notebooks directory")
if str(notebooks_dir.resolve()) not in sys.path:
    sys.path.insert(0, str(notebooks_dir.resolve()))

from _shared.core.lfsr import polynomial_terms  # noqa: E402
from _shared.presentation.widgets import LLREditor  # noqa: E402
from IPython.display import display  # noqa: E402

P = polynomial_terms((0, 2, 5))  # Distinct nonzero exponents, including 0 and K.
# For a degree-32 experiment, supply your chosen degree-32 polynomial here.
K = P[-1]
N = 16  # Choose N > K if you want local parity checks and redundancy.
T = 20
SEED = None  # Use an integer to reproduce the random word and subsequent noise draws.

## Edit the observation $y'$

- Click a bit of $x$ to flip it, or **New random x** to sample a new word. Every change of $x$ resets $y'$ to the newly encoded reference. Changing $N$ also re-encodes and resets; changing $T$ preserves the observation.
- Hold the left mouse button and move over $y'$ cells. A brush acts immediately on entry and repeats every **100 ms** while held over that cell. Keyboard users can focus a cell and press Space or Enter.
- **Flip:** subtract $a s_i$ per tick. This gradually crosses zero; it is not instantaneous sign negation.
- **Suppress:** multiply by $10^{-d/20}$ per tick. Here dB is explicitly an **amplitude attenuation convention for the LLR**, not an SNR conversion. Infinite dB erases immediately.
- **Restore:** add $a s_i$ per tick. It can increase confidence above one. **Reset** instead sets all cells exactly to $s$.
- **Add noise:** add independent $\mathcal N(0,\sigma^2)$ samples to the current $y'$. Repeated clicks accumulate noise. The global control sets $\sigma$ directly in LLR units (default: 0.5 LLR, step: 0.1, minimum: 0).

Green means the sign agrees with the reference; red means it disagrees. Zero is white. Color strength is $1-\exp(-|L|/2)$, on a fixed scale shared with the evolution plot. Hover for the full numeric value.

There is no fixed upper cap on $N$ or $T$. Large words still require rendering and transmitting $N$ editable cells, so the browser can become the bottleneck. Changing $T$ only sets the next explicit run length; it never starts a computation.

In [2]:
# Recreating the editor resets the experiment input; it does not decode.
if "editor" in globals():
    globals()["editor"].close()
import importlib

LLREditor = importlib.reload(sys.modules[LLREditor.__module__]).LLREditor
editor = LLREditor(n=N, iterations=T, terms=P, seed=SEED)
display(editor)

## Synchronous additive soft-XOR iterations

Let $\mathcal C_i$ be the fully contained polynomial checks containing bit $i$. Start with the edited observation $L^{(0)}=y'$. Each iteration is

$$m_{C\to i}^{(t)}=2\operatorname{atanh}\!\left[\prod_{j\in C\setminus\{i\}}\tanh\!\left(L_j^{(t)}/2\right)\right],$$
$$L_i^{(t+1)}=L_i^{(t)}+\sum_{C\in\mathcal C_i}m_{C\to i}^{(t)}.$$

**Every message reads only the previous iteration.** The decoder never reads the true word. Truth is used only by the editor and diagnostics. A check with an erased neighbor sends zero; a bit with no checks stays unchanged.

The implementation evaluates the tanh soft-XOR through an algebraically equivalent, numerically stable pairwise box-plus identity. This avoids rounding `tanh` to exactly +/-1 at high confidence and does not clip LLRs. It uses no damping, normalization, channel reinjection, or early stopping.

This is the requested additive experiment, **not extrinsic belief propagation**: information is reused through cycles and can reinforce incorrect beliefs. Growing magnitude is not by itself evidence of successful correction. Each explicit decoding run starts a fresh trajectory from a copy of the current observation; decoding never overwrites the editor. Later edits cannot change an existing result.

The encoder, decoder and metric loops use `numba.njit(cache=True)`. The first call may compile a kernel; later calls reuse compiled code. The sparse offsets are runtime data, so changing the degree does not require enumerating states or specializing code to a particular polynomial.

For each check, prefix/suffix box-plus folds compute every leave-one-out opinion in $O(|P|)$ work. All reads still come from the previous row. No `fastmath` is used, preserving the zero/erasure and finite-value semantics.

Regrouping floating-point sums changes rounding slightly. Short trajectories agree with the direct tanh formula to numerical precision; long self-reinforcing trajectories can amplify those differences. The update rule itself is unchanged.

## Decode the current observation: head-to-head

Edit `DECODERS` below to choose the configurations to compare. It is an ordered dictionary: **decoder label -> decoder callable**. `functools.partial` binds architecture-specific options, so adding another architecture does not require modifying the runner or the plots.

In [3]:
import importlib
from functools import partial
from time import perf_counter

import _shared.application.comparison as comparison
import _shared.core.decoders._adt as softxor
from _shared.core.decoders import avail_softmajvote

# Reload implementations without recreating the editor or changing its observation.
softxor = importlib.reload(softxor)
avail_softmajvote = importlib.reload(avail_softmajvote)
comparison = importlib.reload(comparison)

DECODERS = {
    "avail-softmajvote(tanh)": partial(
        avail_softmajvote.decode, softxor=softxor.Tanh()
    ),
    "avail-softmajvote(sqrt-sign)": partial(
        avail_softmajvote.decode, softxor=softxor.SqrtSign()
    ),
    "avail-softmajvote(min-sum(0.8))": partial(
        avail_softmajvote.decode,
        softxor=softxor.NormalizedMinSum(coefficient=0.8),
    ),
}

snapshot = editor.snapshot()
true = snapshot["truth"]
terms = snapshot["terms"]
started = perf_counter()
histories = comparison.run_decoders(
    DECODERS,
    snapshot["initial"],
    terms=terms,
    iterations=snapshot["iterations"],
)
print(
    f"Compared {len(histories)} decoders: N={len(true)}, K={terms[-1]}, "
    f"T={snapshot['iterations']} in {perf_counter() - started:.3f} s"
)

Compared 3 decoders: N=16, K=5, T=20 in 0.017 s


## Iteration versus metrics

All plots include $t=0$ and the following $T$ iterations. Selected curves share one plot and their original numerical scale. The **Decoders** and **Metrics** groups independently select which combinations are visible. The three rate metrics are selected initially; loss and signed margin can be enabled separately to avoid compressing the rates against their larger scale. Color identifies the metric; line style identifies the decoder. The rate metrics are:

- **Sign error rate:** fraction of negative signed margins, with exact zero counted as half an error.
- **Erasure fraction:** fraction of exactly zero LLRs.
- **Unsatisfied checks:** hard-decision syndrome fraction (zero LLR is decoded as bit 0). This metric is undefined, shown as a gap, if no checks exist. Another valid codeword can have zero syndrome and still be wrong.

The same plot also shows **mean logistic loss** $\frac1N\sum_i\log(1+e^{-s_iL_i})$, sensitive to confidently wrong decisions, and **mean signed margin** $\frac1N\sum_i s_iL_i$, measuring average alignment and confidence. The latter can hide individual failures; inspect the per-bit plot too.

Drag to zoom, use the mouse wheel to zoom, choose pan in the toolbar, and double-click to reset axes. Use the legend entries to the right of the chart to show or hide curves; All/None acts on one group. Changing a selection fits the axes to visible data. Editing the LLR widget does not touch these plots. Selectors only change visibility of prepared curves; decoder outputs remain unchanged.

In [4]:
import importlib

import _shared.application.metrics as diagnostics
import _shared.presentation.plot_controls as plot_controls
import _shared.presentation.plots as plots

diagnostics = importlib.reload(diagnostics)
plot_controls = importlib.reload(plot_controls)
plots = importlib.reload(plots)
metrics_view = plots.compare(
    histories,
    true,
    kind="metrics",
    terms=terms,
    previous=globals().get("metrics_view"),
)
display(metrics_view)

ComparisonView(children=(FigureWidget({
    'data': [{'hovertemplate': 'Iteration %{x}<br>%{y:.6g}<extra>%{ful…

## Iteration versus bits

One trace per transmitted bit with the following quantity

$$q_i^{(t)}=(-1)^{y_i} L_i^{(t)}=(1-2y_i)L_i^{(t)}.$$

Positive is correct, negative is wrong, and zero is erased. Use the **Bits** and **Decoders** groups to select any subset of bits and decoders. Initially all bits of the first decoder are shown. Color identifies the bit; line style identifies the decoder.

In [5]:
import importlib

import _shared.presentation.plot_controls as plot_controls
import _shared.presentation.plots as plots

plot_controls = importlib.reload(plot_controls)
plots = importlib.reload(plots)
bits_view = plots.compare(
    histories,
    true,
    kind="bits",
    previous=globals().get("bits_view"),
)
display(bits_view)

ComparisonView(children=(FigureWidget({
    'data': [{'hovertemplate': 'Iteration %{x}<br>%{y:.6g}<extra>%{ful…

## Evolution of $y'$

Each row is the complete LLR vector at the next iteration, starting at $t=0$ at the top. Color uses the same truth-relative mapping as the editor. Hover reveals the **raw signed LLR**, not the color-transformed value. The fixed saturating color scale makes low-confidence details visible even when later iterations become very confident.

Select one decoder to the right of the heatmap to display its stored history. The color scale stays fixed across decoders; switching never reruns decoding.


In [6]:
import importlib

import _shared.presentation.plot_controls as plot_controls
import _shared.presentation.plots as plots

plot_controls = importlib.reload(plot_controls)
plots = importlib.reload(plots)
evolution_view = plots.compare(
    histories,
    true,
    kind="evolution",
    previous=globals().get("evolution_view"),
)
display(evolution_view)

ComparisonView(children=(FigureWidget({
    'data': [{'colorbar': {'len': 0.92,
                           'ou…